# O-RAN Resource Allocation — Version 3: Receiver-Aware QoS-Constrained ML/DRL

V3 extends V2 with MRC/ZF/MMSE receiver-aware SINR, a numerical QoS feasibility projection, separate PPO agents per receiver, and independent-seed confidence intervals.

QoS projection uses SLSQP on a non-convex rate-feasibility problem. It is a local numerical attempt, not a guarantee. PPO superiority is not assumed.

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT=Path.cwd()
V3=ROOT/'experiments'/'AI-Enabled-O-RAN-ML-DRL-QoS-Optimization-Version3'
if not V3.exists(): raise FileNotFoundError('Expected Version 3 package under experiments/.')
os.chdir(V3); sys.path.insert(0,str(V3))
print('Version 3 path:',V3)


## Install dependencies and run tests

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)


## Train the supervised traffic predictor

The MLP is compared against a persistence forecast on a synthetic held-out split. Repeat across training seeds before claiming general prediction value.

In [ ]:
subprocess.run([sys.executable,'-m','src.train_traffic'],check=True)


## Train receiver-specific PPO agents

In [ ]:
for receiver in ('MRC','ZF','MMSE'):
    subprocess.run([sys.executable,'-m','src.train_ppo','--receiver',receiver],check=True)


## Evaluate over independent seeds

Five test seeds and 50 blocks are a starting reproducibility check. Strong research conclusions require more training and test seeds, and confidence intervals.

In [ ]:
subprocess.run([sys.executable,'-m','src.evaluate','--seeds','5','--blocks','50'],check=True)


In [ ]:
import pandas as pd
per_seed=pd.read_csv('results/per_seed.csv')
summary=pd.read_csv('results/summary_ci95.csv')
display(per_seed.head()); display(summary)
